# Evaluation Example: Beat Tracking

In [ ]:
import IPython.display as ipd
import matplotlib.pyplot as plt
import librosa.display
import mir_eval
import numpy

from mirdotcom import mirdotcom

mirdotcom.init()

[Documentation: `mir_eval.beat`](http://craffel.github.io/mir_eval/#module-mir_eval.beat)

Evaluation method: compute the error between the estimated beat times and some reference list of beat locations. Many metrics additionally compare the beat sequences at different metric levels in order to deal with the ambiguity of tempo.

Let's evaluate a beat detector on the following audio:

In [ ]:
filename = mirdotcom.get_audio("prelude_cmaj.wav")
y, sr = librosa.load(filename)

In [ ]:
ipd.Audio(y, rate=sr)

## Detect Beats

Estimate the beats using `beat_track`:

In [ ]:
est_tempo, est_beats = librosa.beat.beat_track(y=y, sr=sr, bpm=120)
est_beats = librosa.frames_to_time(est_beats, sr=sr)

In [ ]:
est_beats

Load a fictional reference annotation.

In [ ]:
ref_beats = numpy.array(
    [
        0,
        0.50,
        1.02,
        1.53,
        1.99,
        2.48,
        2.97,
        3.43,
        3.90,
        4.41,
        4.89,
        5.38,
        5.85,
        6.33,
        6.82,
        7.29,
        7.70,
    ]
)

Plot the estimated and reference beats together.

In [ ]:
D = librosa.stft(y)
S = abs(D)
S_db = librosa.amplitude_to_db(S)
librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="log")
plt.ylim(0, 8192)
plt.vlines(est_beats, 0, 8192, color="#00ff00")
plt.scatter(ref_beats, 5000 * numpy.ones_like(ref_beats), color="k", s=100)

## Evaluate

Evaluate using [`mir_eval.beat.evaluate`](https://github.com/craffel/mir_eval/blob/master/mir_eval/beat.py#L704):

In [ ]:
mir_eval.beat.evaluate(ref_beats, est_beats)

Hidden benefits

- Input validation! Many errors can be traced back to ill-formatted data.
- Standardized behavior, full test coverage.

## More than metrics

mir_eval has tools for display and sonification.

In [ ]:
import librosa.display
import mir_eval.display

Common plots: `events`, `labeled_intervals`

pitch, multipitch, piano_roll
segments, hierarchy,
separation

### Example: Events

In [ ]:
librosa.display.specshow(S, x_axis="time", y_axis="mel")
mir_eval.display.events(ref_beats, color="w", alpha=0.8, linewidth=3)
mir_eval.display.events(est_beats, color="c", alpha=0.8, linewidth=3, linestyle="--")

### Example: Labeled Intervals

### Example: Source Separation

In [ ]:
y_harm, y_perc = librosa.effects.hpss(y, margin=8)

In [ ]:
plt.figure(figsize=(12, 4))
mir_eval.display.separation([y_perc, y_harm], sr, labels=["percussive", "harmonic"])
plt.legend()